# Predicting NBA Success Based on Rookie Stats

This notebook represents my workflow for getting data from database in usable format for modeling. There will be another notebook for modeling workflow and feature selection.

Author: Rasheed Martin

In [92]:
import sys
sys.path.append('/Users/rashe/PycharmProjects/PredictingNBACareers')
from utils.duckdb import get_duckdb_connection
import duckdb
import polars as pl

## Let's first get the data for players

### Get NBA Players First Three Season. (Context, we only have data from 2000-2025 so we may have players who started before that but we are treating their first years in the 2000s as their first year)

I will also be able to get their third year stats. So we will do rookie stats, aggregated first three years and third year stats.

In [93]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
WITH player_season_counts AS (
    SELECT
        player_id,
        COUNT(DISTINCT season_id) AS season_count
    FROM player_stat_values
    GROUP BY player_id
    
),
first_three_seasons AS (
    SELECT
        psv.player_id,
        s.label,
        ROW_NUMBER() OVER (
            PARTITION BY psv.player_id
            ORDER BY s.label
        ) AS season_rank
    FROM player_stat_values psv
    JOIN player_season_counts psc
      ON psv.player_id = psc.player_id
     JOIN seasons s on psv.season_id = s.season_id
    GROUP BY psv.player_id, s.label
),
selected_stats AS (
    SELECT
        psv.*
    FROM player_stat_values psv
    JOIN seasons s on psv.season_id = s.season_id
    JOIN first_three_seasons fts
      ON psv.player_id = fts.player_id
     AND s.label = fts.label
    WHERE fts.season_rank <= 3
)
SELECT 
    p.name AS Player,
    s.label AS Season,
    t.abbreviation AS Team,
    st.name AS Stat_Type,
    ps.name AS Stat_Name,
    selected_stats.value AS Stat_Value
FROM selected_stats
JOIN players p ON selected_stats.player_id = p.player_id
JOIN seasons s ON selected_stats.season_id = s.season_id
JOIN teams t ON selected_stats.team_id = t.team_id
JOIN stat_types st ON selected_stats.stat_type_id = st.stat_type_id
JOIN stats ps ON selected_stats.stat_id = ps.stat_id
WHERE st.stat_type_id in (2, 3)
ORDER BY p.name, s.label, stat_name;
        """
    ).arrow()
    df_first_three_years = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [94]:
df_first_three_years

Player,Season,Team,Stat_Type,Stat_Name,Stat_Value
str,i32,str,str,str,f32
"""AC. Green""",2000,"""MIA""","""Total_Player_Data""","""2P""",144.0
"""AC. Green""",2000,"""MIA""","""Total_Player_Data""","""2P%""",0.453
"""AC. Green""",2000,"""MIA""","""Total_Player_Data""","""2PA""",318.0
"""AC. Green""",2000,"""MIA""","""Total_Player_Data""","""3P""",0.0
"""AC. Green""",2000,"""MIA""","""Total_Player_Data""","""3P%""",0.0
…,…,…,…,…,…
"""Zyon Pullin""",2024,"""MEM""","""Advanced_Data""","""USG%""",13.4
"""Zyon Pullin""",2024,"""MEM""","""Advanced_Data""","""VORP""",0.0
"""Zyon Pullin""",2024,"""MEM""","""Advanced_Data""","""WS""",0.0


In [95]:
df_first_three_years = df_first_three_years.pivot(
    index=["Player", "Season"],
    on="Stat_Name",
    values="Stat_Value",
    aggregate_function="first"
)
df_first_three_years

Player,Season,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,Player_Age,STL,STL%,TOV,TOV%,TRB,TRB%,TS%,Trp-Dbl,USG%,VORP,WS,WS/48,eFG%
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",2000,144.0,0.453,318.0,0.0,0.0,6.0,0.019,39.0,4.8,8.0,0.4,-2.2,0.0,206.0,17.299999,2.1,144.0,0.444,324.0,79.0,0.712,111.0,0.343,82.0,1.0,1411.0,-2.2,107.0,9.0,1.1,11.2,119.0,367.0,37.0,30.0,1.2,45.0,10.8,313.0,13.2,0.492,0.0,14.4,-0.1,3.2,0.11,0.444
"""AJ Griffin""",2022,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,19.0,42.0,1.4,42.0,7.1,153.0,6.0,0.577,0.0,17.9,0.4,2.2,0.075,0.56
"""AJ Griffin""",2023,8.0,0.348,23.0,10.0,0.256,39.0,0.629,5.0,3.7,2.0,1.1,-9.6,-3.5,16.0,10.6,0.0,18.0,0.29,62.0,2.0,1.0,2.0,0.032,20.0,0.0,171.0,-6.0,2.0,1.2,-0.4,1.2,6.0,48.0,20.0,1.0,0.3,8.0,11.3,18.0,5.7,0.382,0.0,17.299999,-0.3,-0.4,-0.12,0.371
"""AJ Johnson""",2024,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,20.0,12.0,0.9,35.0,13.2,59.0,4.8,0.48,0.0,17.4,-1.0,-0.4,-0.032,0.441
"""AJ. Green""",2022,9.0,0.45,20.0,44.0,0.419,105.0,0.84,22.0,8.7,0.0,0.0,-0.9,-0.6,39.0,11.3,0.3,53.0,0.424,125.0,4.0,1.0,4.0,0.032,35.0,1.0,345.0,-0.3,6.0,1.9,0.5,11.5,31.0,154.0,23.0,6.0,0.8,9.0,6.6,45.0,6.8,0.607,0.0,16.6,0.1,0.8,0.111,0.6
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zydrunas Ilgauskas""",2001,241.0,0.429,562.0,0.0,0.0,5.0,0.009,70.0,9.4,84.0,4.6,-2.9,-1.6,198.0,17.299999,1.0,241.0,0.425,567.0,208.0,0.754,276.0,0.487,62.0,23.0,1329.0,-1.3,136.0,12.1,1.5,17.4,187.0,690.0,26.0,17.0,0.7,94.0,12.0,334.0,14.8,0.501,0.0,27.0,-0.3,2.5,0.089,0.425
"""Zydrunas Ilgauskas""",2002,495.0,0.443,1117.0,0.0,0.0,5.0,0.004,127.0,10.1,152.0,4.4,-0.3,-0.8,371.0,17.4,2.1,495.0,0.441,1122.0,400.0,0.781,512.0,0.456,81.0,81.0,2432.0,0.5,240.0,10.8,2.9,19.4,274.0,1390.0,27.0,56.0,1.2,210.0,13.5,611.0,14.1,0.516,0.0,27.799999,1.0,5.0,0.099,0.441
"""Zylan Cheatham""",2019,6.0,0.75,8.0,0.0,0.0,1.0,0.111,3.0,7.7,1.0,1.6,-7.2,-0.5,6.0,12.5,0.0,6.0,0.667,9.0,0.0,null,0.0,0.0,4.0,0.0,51.0,-6.6,3.0,6.2,0.0,6.7,10.0,12.0,24.0,1.0,0.9,4.0,30.799999,9.0,9.4,0.667,0.0,10.4,-0.1,0.0,0.034,0.667


Going to filter out rookies and other players who did not play 3 or more years

In [96]:
# players_who_played_first_three_years = df_first_three_years.select("Player").unique().to_series().to_list()

Below we calculate the percent growth from year 1 to year 2 and year 2 to year 3 and then the avg annual year over year growth. The resulting data frame will have their rookie stats included

### Getting general players roy info, all rookie team info, draft pick, draft position

In [97]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
SELECT 
    p.name AS Player,
    p.draft_pick AS Draft_Pick,
    p.draft_class AS Draft_Class,
    r.rank AS ROY_Rank,
    r.first_place_votes AS ROY_Votes,
    r.point_share AS ROY_Point_Share,
    a.rank AS All_Rookie_Rank,
    a.first_place_votes AS All_Rookie_First_Place_Votes,
    a.second_place_votes AS All_Rookie_Second_Place_Votes,
    a.points_share AS All_Rookie_Point_Share

FROM nba_data.main.players p
LEFT JOIN nba_data.main.roy_voting r on p.player_id = r.player_id 
LEFT JOIN nba_data.main.all_rookie_voting a on p.player_id = a.player_id
        """
    ).arrow()
    df_players_rookie_awards = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [98]:
df_players_rookie_awards

Player,Draft_Pick,Draft_Class,ROY_Rank,ROY_Votes,ROY_Point_Share,All_Rookie_Rank,All_Rookie_First_Place_Votes,All_Rookie_Second_Place_Votes,All_Rookie_Point_Share
str,i32,i32,i32,i32,f32,str,i32,i32,f32
"""Nikola Jokic""",41,2014,3,0,0.091,"""1ST Team""",73,40,0.715
"""Kemba Walker""",9,2011,11,0,0.005,"""Received Votes""",0,7,0.121
"""Andrew Bogut""",1,2005,3,0,0.157,"""1ST Team""",26,3,0.948
"""Marc Gasol""",48,2007,8,0,0.007,"""2ND Team""",2,21,0.431
"""Jaylen Brown""",3,2016,8,0,0.002,"""2ND Team""",24,58,0.53
…,…,…,…,…,…,…,…,…,…
"""Walter Herrmann""",null,null,10,0,0.008,"""2ND Team""",1,8,0.172
"""Jorge Garbajosa""",null,null,8,0,0.027,"""1ST Team""",13,11,0.638
"""Juan Carlos Navarro""",40,2002,6,0,0.005,"""2ND Team""",5,14,0.414


Let's merge their rookie awards with the pct differential and rookie stats

In [99]:
import polars as pl

# List your stat columns
stat_cols = df_first_three_years.select(pl.exclude(["Player", "Season", "Player_Age"])).columns

# Build list of aggregate expressions
exprs = []

for stat in stat_cols:
    col = pl.col(stat)
    exprs.extend([
        # Year‑1→2 growth
        ((col.shift(-1).over("Player") - col) / col.shift(-1).over("Player") * 100)
            .alias(f"{stat}_growth_y1_y2"),
        # Year‑2→3 growth
        ((col.shift(-2).over("Player") - col.shift(-1).over("Player"))
         / col.shift(-2).over("Player") * 100)
            .alias(f"{stat}_growth_y2_y3"),
        # Average of the two
        ((( (col.shift(-1).over("Player") - col) / col.shift(-1).over("Player") * 100 )
          + ( (col.shift(-2).over("Player") - col.shift(-1).over("Player")) / col.shift(-2).over("Player") * 100 )
         ) / 2).alias(f"{stat}_avg_yoy_growth")
    ])

# Apply to only first-year rows per player
df_pct_diff_three_years = (
    df_first_three_years.select(["Player", "Season", "Player_Age"] + stat_cols)
    .sort(["Player", "Season"])
    .fill_null(0)
    .with_columns(exprs)
    .filter(pl.col("Season") == pl.col("Season").min().over("Player"))  # keep only the year‑1 row
)

df_pct_diff_three_years

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,STL_avg_yoy_growth,STL%_growth_y1_y2,STL%_growth_y2_y3,STL%_avg_yoy_growth,TOV_growth_y1_y2,TOV_growth_y2_y3,TOV_avg_yoy_growth,TOV%_growth_y1_y2,TOV%_growth_y2_y3,TOV%_avg_yoy_growth,TRB_growth_y1_y2,TRB_growth_y2_y3,TRB_avg_yoy_growth,TRB%_growth_y1_y2,TRB%_growth_y2_y3,TRB%_avg_yoy_growth,TS%_growth_y1_y2,TS%_growth_y2_y3,TS%_avg_yoy_growth,Trp-Dbl_growth_y1_y2,Trp-Dbl_growth_y2_y3,Trp-Dbl_avg_yoy_growth,USG%_growth_y1_y2,USG%_growth_y2_y3,USG%_avg_yoy_growth,VORP_growth_y1_y2,VORP_growth_y2_y3,VORP_avg_yoy_growth,WS_growth_y1_y2,WS_growth_y2_y3,WS_avg_yoy_growth,WS/48_growth_y1_y2,WS/48_growth_y2_y3,WS/48_avg_yoy_growth,eFG%_growth_y1_y2,eFG%_growth_y2_y3,eFG%_avg_yoy_growth
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",2000,37.0,144.0,0.453,318.0,0.0,0.0,6.0,0.019,39.0,4.8,8.0,0.4,-2.2,0.0,206.0,17.299999,2.1,144.0,0.444,324.0,79.0,0.712,111.0,0.343,82.0,1.0,1411.0,-2.2,107.0,9.0,1.1,11.2,119.0,367.0,30.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,null,-366.666626,null,null,-425.0,null,null,37.16814,null,null,-750.0,null,null,-5.263161,null,null,-51.047127,null,null,NaN,null,null,-3.468211,null,null,233.333344,null,null,650.000061,null,null,162.5,null,null,-50.943398,null,null
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""AJ. Green""",2022,23.0,9.0,0.45,20.0,44.0,0.419,105.0,0.84,22.0,8.7,0.0,0.0,-0.9,-0.6,39.0,11.3,0.3,53.0,0.424,125.0,4.0,1.0,4.0,0.032,35.0,1.0,345.0,-0.3,6.0,1.9,0.5,11.5,31.0,154.0,6.0,…,54.504505,-14.285718,36.363636,11.03896,25.0,70.0,47.5,-19.999998,34.523808,7.261905,29.6875,63.218391,46.452946,-19.298252,0.0,-9.649126,1.620744,0.644124,1.132434,NaN,NaN,NaN,-9.21053,-20.634914,-14.922722,-inf,100.0,-inf,27.272728,60.714287,43.993507,-32.14286,-2.43902,-17.290939,-0.166953,2.124185,0.978616
"""AJ. Guyton""",2000,22.0,51.0,0.415,123.0,27.0,0.391,69.0,0.359,64.0,18.1,5.0,0.6,-4.0,-3.2,26.0,5.1,-0.1,78.0,0.406,192.0,15.0,0.833,18.0,0.094,33.0,8.0,630.0,-0.8,10.0,1.8,0.4,10.3,35.0,198.0,9.0,…,-445.0,11.111108,84.210526,47.660816,35.135136,-3600.0,-1782.432373,15.079369,36.999996,26.039682,18.181818,-inf,-inf,19.047613,-inf,-inf,-3.773587,-inf,-inf,NaN,NaN,NaN,25.675678,5.932202,15.80394,400.0,-inf,-inf,-200.000031,200.0,-0.000015,-233.333328,101.333328,-66.0,-4.835162,-inf,-inf
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53

In [100]:
df_pct_diff_rookie_stats = df_pct_diff_three_years.join(
    df_players_rookie_awards,
    on="Player",
    how="left"
)
df_pct_diff_rookie_stats

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,TOV%_avg_yoy_growth,TRB_growth_y1_y2,TRB_growth_y2_y3,TRB_avg_yoy_growth,TRB%_growth_y1_y2,TRB%_growth_y2_y3,TRB%_avg_yoy_growth,TS%_growth_y1_y2,TS%_growth_y2_y3,TS%_avg_yoy_growth,Trp-Dbl_growth_y1_y2,Trp-Dbl_growth_y2_y3,Trp-Dbl_avg_yoy_growth,USG%_growth_y1_y2,USG%_growth_y2_y3,USG%_avg_yoy_growth,VORP_growth_y1_y2,VORP_growth_y2_y3,VORP_avg_yoy_growth,WS_growth_y1_y2,WS_growth_y2_y3,WS_avg_yoy_growth,WS/48_growth_y1_y2,WS/48_growth_y2_y3,WS/48_avg_yoy_growth,eFG%_growth_y1_y2,eFG%_growth_y2_y3,eFG%_avg_yoy_growth,Draft_Pick,Draft_Class,ROY_Rank,ROY_Votes,ROY_Point_Share,All_Rookie_Rank,All_Rookie_First_Place_Votes,All_Rookie_Second_Place_Votes,All_Rookie_Point_Share
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,i32,i32,i32,f32,str,i32,i32,f32
"""AC. Green""",2000,37.0,144.0,0.453,318.0,0.0,0.0,6.0,0.019,39.0,4.8,8.0,0.4,-2.2,0.0,206.0,17.299999,2.1,144.0,0.444,324.0,79.0,0.712,111.0,0.343,82.0,1.0,1411.0,-2.2,107.0,9.0,1.1,11.2,119.0,367.0,30.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,null,-750.0,null,null,-5.263161,null,null,-51.047127,null,null,NaN,null,null,-3.468211,null,null,233.333344,null,null,650.000061,null,null,162.5,null,null,-50.943398,null,null,16,2022,null,null,null,"""Received Votes""",0,26,0.13
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,23,2024,null,null,null,null,null,null,null
"""AJ. Green""",2022,23.0,9.0,0.45,20.0,44.0,0.419,105.0,0.84,22.0,8.7,0.0,0.0,-0.9,-0.6,39.0,11.3,0.3,53.0,0.424,125.0,4.0,1.0,4.0,0.032,35.0,1.0,345.0,-0.3,6.0,1.9,0.5,11.5,31.0,154.0,6.0,…,7.261905,29.6875,63.218391,46.452946,-19.298252,0.0,-9.649126,1.620744,0.644124,1.132434,NaN,NaN,NaN,-9.21053,-20.634914,-14.922722,-inf,100.0,-inf,27.272728,60.714287,43.993507,-32.14286,-2.43902,-17.290939,-0.166953,2.124185,0.978616,null,null,null,null,null,null,null,null,null
"""AJ. Guyton""",2000,22.0,51.0,0.415,123.0,27.0,0.391,69.0,0.359,64.0,18.1,5.0,0.6,-4.0,-3.2,26.0,5.1,-0.1,78.0,0.406,192.0,15.0,0.833,18.0,0.094,33.0,8.0,630.0,-0.8,10.0,1.8,0.4,10.3,35.0,198.0,9.0,…,26.039682,18.181818,-inf,-inf,19.047613,-inf,-inf,-3.773587,-inf,-inf,NaN,NaN,NaN,25.675678,5.932202,15.80394,400.0,-inf,-inf,-200.000031,200.0,-0.000015,-233.333328,101.333328,-66.0,-4.835162,-inf,-inf,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53.0,0.411,129.0,38.0,0.633,60.0,0.465,49.0,1.0,473.0,-3.3,14.0,3.5,0.1,10.0,70.0,153.0,13

Do some feature creation

### Getting team data to see if they made the playoffs prior

In [101]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
WITH player_seasons AS (
    SELECT DISTINCT
        psv.player_id,
        psv.team_id,
        psv.season_id
    FROM player_stat_values psv
)

SELECT 
    p.name AS player_name,
    s.label AS season,
    t.name AS team,
    CASE 
        WHEN tsi_prev.made_playoffs IS NULL THEN NULL
        ELSE tsi_prev.made_playoffs
    END AS team_made_playoffs_last_year
FROM player_seasons ps
JOIN players p ON p.player_id = ps.player_id
JOIN teams t ON t.team_id = ps.team_id
JOIN seasons s ON s.season_id = ps.season_id
JOIN seasons s_prev ON s_prev.label = s.label - 1
LEFT JOIN team_season_info tsi_prev 
    ON tsi_prev.team_id = ps.team_id 
    AND tsi_prev.season_id = s_prev.season_id
        """
    ).arrow()
    df_team_playoffs_prior = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [102]:
df_team_playoffs_prior = df_team_playoffs_prior.filter(pl.col("season") == pl.col("season").min().over("player_name")) # only rookies' first season

# Step 1: Count unique teams per player per season
team_counts = (
    df_team_playoffs_prior.group_by(["player_name", "season"])
      .agg(pl.col("team").n_unique().alias("team_count"))
)

# Step 2: Join this count back to the original DataFrame
df_team_playoffs_prior = df_team_playoffs_prior.join(team_counts, on=["player_name", "season"])

# Step 3: Create a new column with the simplified team info
df_team_playoffs_prior = df_team_playoffs_prior.with_columns([
    pl.when(pl.col("team_count") > 1)
      .then(pl.lit("Multiple Teams"))
      .otherwise(pl.col("team"))
      .alias("team_simplified")
]).drop("team_count")


df_team_playoffs_prior = (
    df_team_playoffs_prior.group_by(["player_name", "season"])
      .agg([
          pl.col("team_simplified").first(),  # or use .max() or .unique() depending on context
          *[
              pl.col(col).first()
              for col in df_team_playoffs_prior.columns
              if col not in ["player_name", "season", "team", "team_simplified"]
          ]
      ])
)

df_team_playoffs_prior = df_team_playoffs_prior.rename({
    "team_simplified": "Team",
    "player_name": "Player",
    "season": "Season"
})

df_team_playoffs_prior


Player,Season,Team,team_made_playoffs_last_year
str,i32,str,bool
"""Linton Johnson""",2003,"""Chicago Bulls""",false
"""Julius Randle""",2014,"""Los Angeles Lakers""",false
"""Dante Cunningham""",2009,"""Portland Trail Blazers""",true
"""Otto Porter Jr""",2013,"""Washington Wizards""",false
"""Ruben Wolkowyski""",2000,"""Seattle SuperSonics""",true
…,…,…,…
"""Lawrence Roberts""",2005,"""Memphis Grizzlies""",true
"""Joe Smith""",2000,"""Detroit Pistons""",true
"""Pablo Prigioni""",2012,"""New York Knicks""",true


In [103]:
df_with_team_player_stats = df_pct_diff_rookie_stats.join(df_team_playoffs_prior, on=["Player", "Season"], how="left")
df_with_team_player_stats

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,TRB_growth_y2_y3,TRB_avg_yoy_growth,TRB%_growth_y1_y2,TRB%_growth_y2_y3,TRB%_avg_yoy_growth,TS%_growth_y1_y2,TS%_growth_y2_y3,TS%_avg_yoy_growth,Trp-Dbl_growth_y1_y2,Trp-Dbl_growth_y2_y3,Trp-Dbl_avg_yoy_growth,USG%_growth_y1_y2,USG%_growth_y2_y3,USG%_avg_yoy_growth,VORP_growth_y1_y2,VORP_growth_y2_y3,VORP_avg_yoy_growth,WS_growth_y1_y2,WS_growth_y2_y3,WS_avg_yoy_growth,WS/48_growth_y1_y2,WS/48_growth_y2_y3,WS/48_avg_yoy_growth,eFG%_growth_y1_y2,eFG%_growth_y2_y3,eFG%_avg_yoy_growth,Draft_Pick,Draft_Class,ROY_Rank,ROY_Votes,ROY_Point_Share,All_Rookie_Rank,All_Rookie_First_Place_Votes,All_Rookie_Second_Place_Votes,All_Rookie_Point_Share,Team,team_made_playoffs_last_year
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,i32,i32,i32,f32,str,i32,i32,f32,str,bool
"""AC. Green""",2000,37.0,144.0,0.453,318.0,0.0,0.0,6.0,0.019,39.0,4.8,8.0,0.4,-2.2,0.0,206.0,17.299999,2.1,144.0,0.444,324.0,79.0,0.712,111.0,0.343,82.0,1.0,1411.0,-2.2,107.0,9.0,1.1,11.2,119.0,367.0,30.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Miami Heat""",true
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,null,null,-5.263161,null,null,-51.047127,null,null,NaN,null,null,-3.468211,null,null,233.333344,null,null,650.000061,null,null,162.5,null,null,-50.943398,null,null,16,2022,null,null,null,"""Received Votes""",0,26,0.13,"""Atlanta Hawks""",true
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,23,2024,null,null,null,null,null,null,null,"""Multiple Teams""",false
"""AJ. Green""",2022,23.0,9.0,0.45,20.0,44.0,0.419,105.0,0.84,22.0,8.7,0.0,0.0,-0.9,-0.6,39.0,11.3,0.3,53.0,0.424,125.0,4.0,1.0,4.0,0.032,35.0,1.0,345.0,-0.3,6.0,1.9,0.5,11.5,31.0,154.0,6.0,…,63.218391,46.452946,-19.298252,0.0,-9.649126,1.620744,0.644124,1.132434,NaN,NaN,NaN,-9.21053,-20.634914,-14.922722,-inf,100.0,-inf,27.272728,60.714287,43.993507,-32.14286,-2.43902,-17.290939,-0.166953,2.124185,0.978616,null,null,null,null,null,null,null,null,null,"""Milwaukee Bucks""",true
"""AJ. Guyton""",2000,22.0,51.0,0.415,123.0,27.0,0.391,69.0,0.359,64.0,18.1,5.0,0.6,-4.0,-3.2,26.0,5.1,-0.1,78.0,0.406,192.0,15.0,0.833,18.0,0.094,33.0,8.0,630.0,-0.8,10.0,1.8,0.4,10.3,35.0,198.0,9.0,…,-inf,-inf,19.047613,-inf,-inf,-3.773587,-inf,-inf,NaN,NaN,NaN,25.675678,5.932202,15.80394,400.0,-inf,-inf,-200.000031,200.0,-0.000015,-233.333328,101.333328,-66.0,-4.835162,-inf,-inf,null,null,null,null,null,null,null,null,null,"""Chicago Bulls""",false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Unknown""",null
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53.0,0.411,129.0,38.0,0.6

### Get third year stats

In [104]:
df_three_year_stats = df_first_three_years.join(
    df_first_three_years.group_by("Player").agg(pl.col("Season").max().alias("Season")),
    on=["Player", "Season"]
)
df_three_year_stats

Player,Season,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,Player_Age,STL,STL%,TOV,TOV%,TRB,TRB%,TS%,Trp-Dbl,USG%,VORP,WS,WS/48,eFG%
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",2000,144.0,0.453,318.0,0.0,0.0,6.0,0.019,39.0,4.8,8.0,0.4,-2.2,0.0,206.0,17.299999,2.1,144.0,0.444,324.0,79.0,0.712,111.0,0.343,82.0,1.0,1411.0,-2.2,107.0,9.0,1.1,11.2,119.0,367.0,37.0,30.0,1.2,45.0,10.8,313.0,13.2,0.492,0.0,14.4,-0.1,3.2,0.11,0.444
"""AJ Griffin""",2023,8.0,0.348,23.0,10.0,0.256,39.0,0.629,5.0,3.7,2.0,1.1,-9.6,-3.5,16.0,10.6,0.0,18.0,0.29,62.0,2.0,1.0,2.0,0.032,20.0,0.0,171.0,-6.0,2.0,1.2,-0.4,1.2,6.0,48.0,20.0,1.0,0.3,8.0,11.3,18.0,5.7,0.382,0.0,17.299999,-0.3,-0.4,-0.12,0.371
"""AJ Johnson""",2024,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,20.0,12.0,0.9,35.0,13.2,59.0,4.8,0.48,0.0,17.4,-1.0,-0.4,-0.032,0.441
"""AJ. Green""",2024,27.0,0.443,61.0,155.0,0.427,363.0,0.856,108.0,8.5,7.0,0.4,-2.2,-0.4,156.0,9.8,1.3,182.0,0.429,424.0,22.0,0.815,27.0,0.064,73.0,7.0,1659.0,-1.7,18.0,1.2,1.6,9.2,157.0,541.0,25.0,37.0,1.1,40.0,8.4,174.0,5.7,0.621,0.0,12.6,-0.1,2.8,0.082,0.612
"""AJ. Guyton""",2002,0.0,0.0,3.0,0.0,0.0,1.0,0.25,2.0,28.700001,0.0,0.0,-16.4,-2.3,0.0,0.0,0.0,0.0,0.0,4.0,0.0,null,0.0,0.0,2.0,0.0,9.0,-14.1,0.0,0.0,-0.1,-9.0,0.0,0.0,24.0,1.0,5.7,1.0,20.0,0.0,0.0,0.0,0.0,23.6,0.0,-0.1,-0.45,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,25.0,2.0,1.4,5.0,13.4,8.0,6.2,0.435,0.0,22.9,-0.1,-0.1,-0.042,0.417
"""Zoran Planinic""",2005,53.0,0.417,127.0,13.0,0.232,56.0,0.306,53.0,15.0,4.0,0.5,-5.7,0.4,60.0,12.0,0.8,66.0,0.361,183.0,46.0,0.697,66.0,0.361,56.0,1.0,596.0,-6.1,14.0,2.7,-0.8,6.9,91.0,191.0,23.0,20.0,1.8,43.0,16.9,74.0,7.3,0.45,0.0,20.200001,-0.6,0.1,0.005,0.396
"""Zydrunas Ilgauskas""",2002,495.0,0.443,1117.0,0.0,0.0,5.0,0.004,127.0,10.1,152.0,4.4,-0.3,-0.8,371.0,17.4,2.1,495.0,0.441,1122.0,400.0,0.781,512.0,0.456,81.0,81.0,2432.0,0.5,240.0,10.8,2.9,19.4,274.0,1390.0,27.0,56.0,1.2,210.0,13.5,611.0,14.1,0.516,0.0,27.799999,1.0,5.0,0.099,0.441


In [105]:
df_without_target = df_with_team_player_stats.join(df_three_year_stats.rename({col: f"{col}_third" for col in df_first_three_years.columns if col not in  ["Player"]}), on=["Player"], how="left")
df_without_target

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,BLK%_third,BPM_third,DBPM_third,DRB_third,DRB%_third,DWS_third,FG_third,FG%_third,FGA_third,FT_third,FT%_third,FTA_third,FTr_third,G_third,GS_third,MP_third,OBPM_third,ORB_third,ORB%_third,OWS_third,PER_third,PF_third,PTS_third,Player_Age_third,STL_third,STL%_third,TOV_third,TOV%_third,TRB_third,TRB%_third,TS%_third,Trp-Dbl_third,USG%_third,VORP_third,WS_third,WS/48_third,eFG%_third
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",2000,37.0,144.0,0.453,318.0,0.0,0.0,6.0,0.019,39.0,4.8,8.0,0.4,-2.2,0.0,206.0,17.299999,2.1,144.0,0.444,324.0,79.0,0.712,111.0,0.343,82.0,1.0,1411.0,-2.2,107.0,9.0,1.1,11.2,119.0,367.0,30.0,…,0.4,-2.2,0.0,206.0,17.299999,2.1,144.0,0.444,324.0,79.0,0.712,111.0,0.343,82.0,1.0,1411.0,-2.2,107.0,9.0,1.1,11.2,119.0,367.0,37.0,30.0,1.2,45.0,10.8,313.0,13.2,0.492,0.0,14.4,-0.1,3.2,0.11,0.444
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,1.1,-9.6,-3.5,16.0,10.6,0.0,18.0,0.29,62.0,2.0,1.0,2.0,0.032,20.0,0.0,171.0,-6.0,2.0,1.2,-0.4,1.2,6.0,48.0,20.0,1.0,0.3,8.0,11.3,18.0,5.7,0.382,0.0,17.299999,-0.3,-0.4,-0.12,0.371
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,20.0,12.0,0.9,35.0,13.2,59.0,4.8,0.48,0.0,17.4,-1.0,-0.4,-0.032,0.441
"""AJ. Green""",2022,23.0,9.0,0.45,20.0,44.0,0.419,105.0,0.84,22.0,8.7,0.0,0.0,-0.9,-0.6,39.0,11.3,0.3,53.0,0.424,125.0,4.0,1.0,4.0,0.032,35.0,1.0,345.0,-0.3,6.0,1.9,0.5,11.5,31.0,154.0,6.0,…,0.4,-2.2,-0.4,156.0,9.8,1.3,182.0,0.429,424.0,22.0,0.815,27.0,0.064,73.0,7.0,1659.0,-1.7,18.0,1.2,1.6,9.2,157.0,541.0,25.0,37.0,1.1,40.0,8.4,174.0,5.7,0.621,0.0,12.6,-0.1,2.8,0.082,0.612
"""AJ. Guyton""",2000,22.0,51.0,0.415,123.0,27.0,0.391,69.0,0.359,64.0,18.1,5.0,0.6,-4.0,-3.2,26.0,5.1,-0.1,78.0,0.406,192.0,15.0,0.833,18.0,0.094,33.0,8.0,630.0,-0.8,10.0,1.8,0.4,10.3,35.0,198.0,9.0,…,0.0,-16.4,-2.3,0.0,0.0,0.0,0.0,0.0,4.0,0.0,null,0.0,0.0,2.0,0.0,9.0,-14.1,0.0,0.0,-0.1,-9.0,0.0,0.0,24.0,1.0,5.7,1.0,20.0,0.0,0.0,0.0,0.0,23.6,0.0,-0.1,-0.45,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,25.0,2.0,1.4,5.0,13.4,8.0,6.2,0.435,0.0,22.9,-0.1,-0.1,-0.042,0.417
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53.0,0.411,129.0,38.0,0.633,60.0,0.465,49.0,1.0,473.0,-3.3,14.0,3.5,0.1,10.0,70.0,153.0,13.0,…,0.5,-5.7,0.4,60.0,12.0,0.8,66.0,0.361,183.0,46.0,0.697,66.0,0.361,56.0,1.0,596.0,-6.1,14.0,2.7,-0.8,6.9,91.0,191.0,23.0,20.0,1.8,43.0,16.9,74.0,7.3,0.45,0.0,20.200001,-0.6,0.1,0.005,0.396
"""Zydrunas Ilgauskas""",2000,25.0,114.0,0.491,232.0,0.0,0.0,2.0,0.009,18.0,5.4,37.0,4.4,-1.8,-0.4,95.0,17.799999,0.7,114.0,0.487,234.0,53.0,0.679,78.0,0.333,24.0,24.0,616.0,-1.4,65.0,12.3,0.1,16.0,78.0,281.0,15.0,…,4.4,-0.3,-0.8,371.0,17.4,2.1,495.0,0.441,1122.0,400.0,0.781,512.0,0.456,81.0,81.0,2432.0,0.5,240.0,10.8,2.9,19.4,2

Looking at the null count, it is obvious based on context that if the players have null values at any step they did not take any or have any. 

From the above analysis, we can see that the following pairs of features have the highest absolute correlation:
Field Goals and Points
2 Points Attempted vs 2 Points Made
Minutes Per Games

In [106]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
WITH player_stats as (
SELECT
    MAX(CASE WHEN st.name = 'WS' THEN psv.value ELSE NULL END) AS ws,
    MAX(CASE WHEN st.name = 'USG%' THEN psv.value ELSE NULL END) AS usage,
	MAX(CASE WHEN st.name = 'G' THEN psv.value ELSE NULL END) AS games_played,
	MAX(CASE WHEN st.name = 'MP' THEN psv.value ELSE NULL END) AS minutes_played,
	MAX(CASE WHEN st.name = 'GS' THEN psv.value ELSE NULL END) AS games_started,
	MAX(CASE WHEN st.name = 'BPM' THEN psv.value ELSE NULL END) AS box_plus_minus,
	psv.season_id,
	psv.player_id,
	psv.team_id
FROM
	nba_data.main.player_stat_values psv
JOIN nba_data.main.stats st on
	st.stat_id = psv.stat_id
JOIN nba_data.main.stat_types s on s.stat_type_id = psv.stat_type_id
WHERE psv.stat_type_id in (3, 2) -- Only Total and Advanced Stats
GROUP BY psv.player_id, psv.season_id, psv.team_id
)
SELECT
 p.name as Player,
 ps.ws as Win_Share,
 ps."usage" as Usage,
 ps.games_played as Games_Played,
 ps.minutes_played as Minutes_Played,
 ps.games_started as Games_Started,
 ps.box_plus_minus as Box_Plus_Minus,
 a.all_nba_team as All_Nba_Team,
 a.allstar as AllStar,
 a.six_moy_rank as Six_MOY,
 s."label" as Season,
 t.name As Team
 
FROM nba_data.main.award_info a
JOIN nba_data.main.players p on a.player_id = p.player_id
JOIN player_stats ps on ps.player_id = p.player_id AND ps.season_id = a.season_id
JOIN nba_data.main.seasons s on s.season_id = ps.season_id
JOIN nba_data.main.teams t on t.team_id = a.team_id
        """
    ).arrow()
    df_award_info = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [107]:
df_award_info

Player,Win_Share,Usage,Games_Played,Minutes_Played,Games_Started,Box_Plus_Minus,All_Nba_Team,AllStar,Six_MOY,Season,Team
str,f32,f32,f32,f32,f32,f32,i32,bool,i32,i32,str
"""Charles Oakley""",-0.5,12.1,57.0,1383.0,36.0,-5.1,0,false,0,2001,"""Chicago Bulls"""
"""Al Harrington""",2.6,21.6,44.0,1313.0,1.0,-1.9,0,false,0,2001,"""Indiana Pacers"""
"""Rafer Alston""",0.7,19.0,50.0,600.0,7.0,-0.7,0,false,0,2001,"""Milwaukee Bucks"""
"""Felton Spencer""",-0.1,9.9,32.0,248.0,8.0,-9.9,0,false,0,2001,"""New York Knicks"""
"""Maurice Evans""",0.1,22.9,10.0,45.0,0.0,-4.7,0,false,0,2001,"""Minnesota Timberwolves"""
…,…,…,…,…,…,…,…,…,…,…,…
"""OG Anunoby""",2.6,17.6,50.0,1702.0,50.0,-0.6,0,false,0,2023,"""Unknown"""
"""Dylan Windler""",0.3,12.5,17.0,108.0,0.0,0.7,0,false,0,2023,"""New York Knicks"""
"""Dennis Schroder""",3.2,21.6,75.0,2105.0,49.0,-2.2,0,false,0,2024,"""Unknown"""


In [108]:
df_award_info.null_count()

Player,Win_Share,Usage,Games_Played,Minutes_Played,Games_Started,Box_Plus_Minus,All_Nba_Team,AllStar,Six_MOY,Season,Team
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,3,0,0,0,3,0,0,0,0,0


In [109]:
def min_max_norm(col: str) -> pl.Expr:
    return ((pl.col(col) - pl.col(col).min()) / (pl.col(col).max() - pl.col(col).min())).alias(f"{col}_norm")

Creating the target variable to model against

In [110]:
# Aggregate stats per player
player_summary = df_award_info.group_by("Player").agg([
    pl.col("All_Nba_Team").max().alias("All_Nba_Team_Max"),
    pl.col("AllStar").max().alias("AllStar_Max"),
    pl.col("Games_Played").mean().alias("Games_Played_Mean"),
    pl.col("Win_Share").sum().alias("Win_Share_Sum"),
    pl.col("Box_Plus_Minus").mean().alias("Box_Plus_Minus_Mean"),
    pl.col("Minutes_Played").mean().alias("Minutes_Played_Mean"),
    pl.col("Games_Started").mean().alias("Games_Started_Mean"),
    pl.col("Usage").mean().alias("Usage_Mean")
])
player_summary = player_summary.with_columns([
    (
        0.5 * min_max_norm("Win_Share_Sum") +
        0.4 * min_max_norm("Box_Plus_Minus_Mean") +
        0.4 * min_max_norm("Usage_Mean") +
        0.2 * min_max_norm("Games_Started_Mean") +
        0.1 * min_max_norm("Minutes_Played_Mean")).alias("Player_Score")
])
# Assign player type
five_tier_label = player_summary.with_columns([
    pl.when(pl.col("All_Nba_Team_Max") > 0)
      .then(pl.lit("All NBA Caliber"))
      .when(pl.col("AllStar_Max") > 0)
      .then(pl.lit("All Star Caliber"))
      .when(
          pl.col("Player_Score") > 0.4
      ).then(pl.lit("Star Player"))
      .when(
          pl.col("Player_Score") > 0.3
      ).then(pl.lit("Role Player"))
      .otherwise(pl.lit("Bust Player"))
      .alias("Player_Type")
])

five_tier_label

Player,All_Nba_Team_Max,AllStar_Max,Games_Played_Mean,Win_Share_Sum,Box_Plus_Minus_Mean,Minutes_Played_Mean,Games_Started_Mean,Usage_Mean,Player_Score,Player_Type
str,i32,bool,f32,f32,f32,f32,f32,f32,f32,str
"""Michael Redd""",3,true,52.416668,55.799999,0.208333,1675.333374,35.75,25.625,0.552164,"""All NBA Caliber"""
"""Jaylen Hoard""",0,false,13.0,1.3,-1.833333,221.0,1.666667,17.4,0.268342,"""Bust Player"""
"""Sean Kilpatrick""",0,false,39.25,2.6,-2.375,770.5,6.25,21.775,0.322987,"""Role Player"""
"""Frank Mason III""",0,false,25.75,1.2,-3.275,404.0,0.75,21.549999,0.290564,"""Bust Player"""
"""Russ Smith""",0,false,13.5,-0.2,-5.1,65.5,0.0,26.85,0.298299,"""Bust Player"""
…,…,…,…,…,…,…,…,…,…,…
"""Tim Hardaway""",0,false,51.666668,8.4,0.966667,1447.0,31.0,22.699999,0.432328,"""Star Player"""
"""Shelvin Mack""",0,false,57.0,10.3,-2.05,1049.0,7.0,19.0625,0.334382,"""Role Player"""
"""Damone Brown""",0,false,9.75,-0.6,-10.55,87.75,0.75,24.174999,0.268541,"""Bust Player"""


In [111]:
from pathlib import Path
path = Path.cwd().parent.parent / "data/clean_data" / "player_label.csv"
five_tier_label.select(["Player", "Player_Type"]).write_csv(path)

In [112]:
player_5tier_label_only_three_years = five_tier_label.join(pl.DataFrame({"Player": players_who_played_first_three_years}), on="Player", how="inner")

In [113]:
path = Path.cwd().parent.parent / "data/clean_data" / "player_5tier_label.csv"
five_tier_label.select(["Player", "Player_Type"]).write_csv(path)

Combine this player_label with our full dataset

In [114]:
path = Path.cwd().parent.parent / "data/refined_raw_data" / "Player_Type_Modeling"
path.mkdir(parents=True, exist_ok=True)
df_without_target.join(
    five_tier_label.rename({"Player_Type": "Player_Label"}),
    on="Player",
    how="left"
).write_csv(
    path / "player_5tier_label_with_target.csv"
)

## Updated due to Experimentation with 5 label classification

In [115]:
# Assign player type
three_tier_label = player_summary.with_columns([
    pl.when(pl.col("Player_Score") > 0.4)
      .then(pl.lit("Star Player"))
      .when(
          pl.col("Player_Score") > 0.3
      ).then(pl.lit("Role Player"))
      .otherwise(pl.lit("Bust Player"))
      .alias("Player_Type")
])
three_tier_label

Player,All_Nba_Team_Max,AllStar_Max,Games_Played_Mean,Win_Share_Sum,Box_Plus_Minus_Mean,Minutes_Played_Mean,Games_Started_Mean,Usage_Mean,Player_Score,Player_Type
str,i32,bool,f32,f32,f32,f32,f32,f32,f32,str
"""Michael Redd""",3,true,52.416668,55.799999,0.208333,1675.333374,35.75,25.625,0.552164,"""Star Player"""
"""Jaylen Hoard""",0,false,13.0,1.3,-1.833333,221.0,1.666667,17.4,0.268342,"""Bust Player"""
"""Sean Kilpatrick""",0,false,39.25,2.6,-2.375,770.5,6.25,21.775,0.322987,"""Role Player"""
"""Frank Mason III""",0,false,25.75,1.2,-3.275,404.0,0.75,21.549999,0.290564,"""Bust Player"""
"""Russ Smith""",0,false,13.5,-0.2,-5.1,65.5,0.0,26.85,0.298299,"""Bust Player"""
…,…,…,…,…,…,…,…,…,…,…
"""Tim Hardaway""",0,false,51.666668,8.4,0.966667,1447.0,31.0,22.699999,0.432328,"""Star Player"""
"""Shelvin Mack""",0,false,57.0,10.3,-2.05,1049.0,7.0,19.0625,0.334382,"""Role Player"""
"""Damone Brown""",0,false,9.75,-0.6,-10.55,87.75,0.75,24.174999,0.268541,"""Bust Player"""


In [116]:
from pathlib import Path
path = Path.cwd().parent.parent / "data/clean_data" / "3_player_label.csv"
three_tier_label.select(["Player", "Player_Type"]).write_csv(path)

In [117]:
player_3tier_label_only_three_years = three_tier_label.join(pl.DataFrame({"Player": players_who_played_first_three_years}), on="Player", how="inner")

In [118]:
player_3tier_label_only_three_years

Player,All_Nba_Team_Max,AllStar_Max,Games_Played_Mean,Win_Share_Sum,Box_Plus_Minus_Mean,Minutes_Played_Mean,Games_Started_Mean,Usage_Mean,Player_Score,Player_Type
str,i32,bool,f32,f32,f32,f32,f32,f32,f32,str
"""Michael Redd""",3,true,52.416668,55.799999,0.208333,1675.333374,35.75,25.625,0.552164,"""Star Player"""
"""Sean Kilpatrick""",0,false,39.25,2.6,-2.375,770.5,6.25,21.775,0.322987,"""Role Player"""
"""Frank Mason III""",0,false,25.75,1.2,-3.275,404.0,0.75,21.549999,0.290564,"""Bust Player"""
"""Goran Dragic""",3,true,62.470589,57.299995,-0.794118,1608.470581,31.17647,22.488235,0.521081,"""Star Player"""
"""DJ. Wilson""",0,false,24.666666,2.9,2.65,303.0,1.0,16.633333,0.282055,"""Bust Player"""
…,…,…,…,…,…,…,…,…,…,…
"""Treveon Graham""",0,false,45.0,2.9,-3.25,720.75,11.0,12.25,0.278177,"""Bust Player"""
"""Ayo Dosunmu""",0,false,69.75,12.5,-1.725,1953.5,38.5,15.700001,0.428346,"""Star Player"""
"""Rick Brunson""",0,false,40.833332,3.2,-2.216667,587.833313,6.833333,15.966667,0.287759,"""Bust Player"""


In [119]:
path = Path.cwd().parent.parent / "data/refined_raw_data" / "Player_Type_Modeling"
path.mkdir(parents=True, exist_ok=True)
df_without_target.join(
    three_tier_label.rename({"Player_Type": "Player_Label"}),
    on="Player",
    how="left"
).write_csv(
    path / "player_3tier_label_with_target_all_years.csv"
)

In [120]:
df_without_target

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,BLK%_third,BPM_third,DBPM_third,DRB_third,DRB%_third,DWS_third,FG_third,FG%_third,FGA_third,FT_third,FT%_third,FTA_third,FTr_third,G_third,GS_third,MP_third,OBPM_third,ORB_third,ORB%_third,OWS_third,PER_third,PF_third,PTS_third,Player_Age_third,STL_third,STL%_third,TOV_third,TOV%_third,TRB_third,TRB%_third,TS%_third,Trp-Dbl_third,USG%_third,VORP_third,WS_third,WS/48_third,eFG%_third
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",2000,37.0,144.0,0.453,318.0,0.0,0.0,6.0,0.019,39.0,4.8,8.0,0.4,-2.2,0.0,206.0,17.299999,2.1,144.0,0.444,324.0,79.0,0.712,111.0,0.343,82.0,1.0,1411.0,-2.2,107.0,9.0,1.1,11.2,119.0,367.0,30.0,…,0.4,-2.2,0.0,206.0,17.299999,2.1,144.0,0.444,324.0,79.0,0.712,111.0,0.343,82.0,1.0,1411.0,-2.2,107.0,9.0,1.1,11.2,119.0,367.0,37.0,30.0,1.2,45.0,10.8,313.0,13.2,0.492,0.0,14.4,-0.1,3.2,0.11,0.444
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,1.1,-9.6,-3.5,16.0,10.6,0.0,18.0,0.29,62.0,2.0,1.0,2.0,0.032,20.0,0.0,171.0,-6.0,2.0,1.2,-0.4,1.2,6.0,48.0,20.0,1.0,0.3,8.0,11.3,18.0,5.7,0.382,0.0,17.299999,-0.3,-0.4,-0.12,0.371
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,20.0,12.0,0.9,35.0,13.2,59.0,4.8,0.48,0.0,17.4,-1.0,-0.4,-0.032,0.441
"""AJ. Green""",2022,23.0,9.0,0.45,20.0,44.0,0.419,105.0,0.84,22.0,8.7,0.0,0.0,-0.9,-0.6,39.0,11.3,0.3,53.0,0.424,125.0,4.0,1.0,4.0,0.032,35.0,1.0,345.0,-0.3,6.0,1.9,0.5,11.5,31.0,154.0,6.0,…,0.4,-2.2,-0.4,156.0,9.8,1.3,182.0,0.429,424.0,22.0,0.815,27.0,0.064,73.0,7.0,1659.0,-1.7,18.0,1.2,1.6,9.2,157.0,541.0,25.0,37.0,1.1,40.0,8.4,174.0,5.7,0.621,0.0,12.6,-0.1,2.8,0.082,0.612
"""AJ. Guyton""",2000,22.0,51.0,0.415,123.0,27.0,0.391,69.0,0.359,64.0,18.1,5.0,0.6,-4.0,-3.2,26.0,5.1,-0.1,78.0,0.406,192.0,15.0,0.833,18.0,0.094,33.0,8.0,630.0,-0.8,10.0,1.8,0.4,10.3,35.0,198.0,9.0,…,0.0,-16.4,-2.3,0.0,0.0,0.0,0.0,0.0,4.0,0.0,null,0.0,0.0,2.0,0.0,9.0,-14.1,0.0,0.0,-0.1,-9.0,0.0,0.0,24.0,1.0,5.7,1.0,20.0,0.0,0.0,0.0,0.0,23.6,0.0,-0.1,-0.45,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,25.0,2.0,1.4,5.0,13.4,8.0,6.2,0.435,0.0,22.9,-0.1,-0.1,-0.042,0.417
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53.0,0.411,129.0,38.0,0.633,60.0,0.465,49.0,1.0,473.0,-3.3,14.0,3.5,0.1,10.0,70.0,153.0,13.0,…,0.5,-5.7,0.4,60.0,12.0,0.8,66.0,0.361,183.0,46.0,0.697,66.0,0.361,56.0,1.0,596.0,-6.1,14.0,2.7,-0.8,6.9,91.0,191.0,23.0,20.0,1.8,43.0,16.9,74.0,7.3,0.45,0.0,20.200001,-0.6,0.1,0.005,0.396
"""Zydrunas Ilgauskas""",2000,25.0,114.0,0.491,232.0,0.0,0.0,2.0,0.009,18.0,5.4,37.0,4.4,-1.8,-0.4,95.0,17.799999,0.7,114.0,0.487,234.0,53.0,0.679,78.0,0.333,24.0,24.0,616.0,-1.4,65.0,12.3,0.1,16.0,78.0,281.0,15.0,…,4.4,-0.3,-0.8,371.0,17.4,2.1,495.0,0.441,1122.0,400.0,0.781,512.0,0.456,81.0,81.0,2432.0,0.5,240.0,10.8,2.9,19.4,2